# COMM117 Week 4 Workshop — Build a Transformer Block (No Training)

**Workshop focus:** implement the **Transformer decoder block** (mechanics only), without training.

What you will build:
1. Causal mask (look-ahead mask)
2. Scaled dot-product attention (SDPA)
3. Multi-head self-attention (packed QKV)
4. Feed-forward network (FFN)
5. **Pre-LN Transformer block** with residual connections

✅ No dataset, no optimization, no training loop.  
✅ Everything is validated using **shape checks + small numerical sanity checks**.

**Submission:** a notebook that passes all tests in the final section.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)


device: cpu


## 0) Set dimensions + dummy input
We use random tensors to test the block wiring.

Notation:
- B = batch size
- N = sequence length
- D = model dimension
- H = number of heads
- Dh = D/H


In [2]:
B, N, D, H = 2, 8, 64, 4
assert D % H == 0
Dh = D // H

x = torch.randn(B, N, D, device=device)
print("x:", x.shape)


x: torch.Size([2, 8, 64])


In [3]:
import math
import torch
import torch.nn.functional as F

def causal_mask(N, device):
    """
    Boolean mask (1,1,N,N). True=allowed, False=blocked future (j > i).
    """
    m = torch.tril(torch.ones(N, N, dtype=torch.bool, device=device))
    return m[None, None, :, :]

def layer_norm(x, w, b, eps=1e-5):
    """
    x: (B,N,D)
    w,b: (D,)
    """
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    xhat = (x - mean) / torch.sqrt(var + eps)
    return xhat * w + b

def sdpa(q, k, v, mask=None):
    """
    q,k,v: (B,H,N,Dh)
    mask: broadcastable to (B,H,N,N) boolean (True=allowed)
    """
    Dh = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(Dh)  # (B,H,N,N)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    out = attn @ v  # (B,H,N,Dh)
    return out, attn

def mha_packed_qkv(x, Wqkv, Wo, H, mask=None):
    """
    x:    (B,N,D)
    Wqkv: (D,3D)  packed projection
    Wo:   (D,D)   output projection
    H:    number of heads
    """
    B, N, D = x.shape
    assert D % H == 0
    Dh = D // H

    # Packed QKV
    qkv = x @ Wqkv                 # (B,N,3D)
    q, k, v = qkv.split(D, dim=-1) # each (B,N,D)

    # Reshape into heads: (B,N,D) -> (B,H,N,Dh)
    q = q.view(B, N, H, Dh).transpose(1, 2)
    k = k.view(B, N, H, Dh).transpose(1, 2)
    v = v.view(B, N, H, Dh).transpose(1, 2)

    ctx, attn = sdpa(q, k, v, mask=mask)  # ctx: (B,H,N,Dh)

    # Concat heads: (B,H,N,Dh) -> (B,N,D)
    ctx = ctx.transpose(1, 2).contiguous().view(B, N, D)

    y = ctx @ Wo  # (B,N,D)
    return y, attn

def ffn(x, W1, b1, W2, b2):
    """
    x: (B,N,D)
    W1: (D,d_ff), b1: (d_ff,)
    W2: (d_ff,D), b2: (D,)
    """
    h = x @ W1 + b1
    h = F.gelu(h)
    out = h @ W2 + b2
    return out

def preln_block(x, ln1_w, ln1_b, ln2_w, ln2_b,
                Wqkv, Wo, H,
                W1, b1, W2, b2,
                mask=None, eps=1e-5):
    """
    Pre-LN block:
      x = x + MHA(LN1(x))
      x = x + FFN(LN2(x))
    Returns: x, attn
    """
    u = layer_norm(x, ln1_w, ln1_b, eps=eps)
    a, attn = mha_packed_qkv(u, Wqkv, Wo, H, mask=mask)
    x = x + a

    v = layer_norm(x, ln2_w, ln2_b, eps=eps)
    x = x + ffn(v, W1, b1, W2, b2)

    return x, attn


In [9]:
import math
import torch

# ---- choose sizes ----
B, N, D, H = 2, 8, 32, 4
d_ff = 4 * D
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- input ----
x = torch.randn(B, N, D, device=device)

# ---- params for preln_block ----
ln1_w = torch.ones(D, device=device)
ln1_b = torch.zeros(D, device=device)
ln2_w = torch.ones(D, device=device)
ln2_b = torch.zeros(D, device=device)

Wqkv = torch.randn(D, 3 * D, device=device) / math.sqrt(D)
Wo   = torch.randn(D, D, device=device) / math.sqrt(D)

W1 = torch.randn(D, d_ff, device=device) / math.sqrt(D)
b1 = torch.zeros(d_ff, device=device)
W2 = torch.randn(d_ff, D, device=device) / math.sqrt(d_ff)
b2 = torch.zeros(D, device=device)

# === RUN TESTS AFTER FILLING TODOS ===
m = causal_mask(N, device=device)   # (1,1,N,N)

y, attn = preln_block(
    x, ln1_w, ln1_b, ln2_w, ln2_b,
    Wqkv, Wo, H,
    W1, b1, W2, b2,
    mask=m
)

# Shape tests
assert y.shape == (B, N, D), f"y shape wrong: {y.shape}"
assert attn.shape == (B, H, N, N), f"attn shape wrong: {attn.shape}"

# Softmax row sums (should be ~1)
row_sums = attn.sum(dim=-1)  # (B,H,N)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4), "attn rows not summing to 1"

# Causal property: for each i, attention on j>i should be ~0
future = torch.triu(torch.ones(N, N, dtype=torch.bool, device=device), diagonal=1)  # j>i
future_mass = attn.masked_select(future[None, None, :, :]).abs().max().item()
print("max attention mass on future positions:", future_mass)
assert future_mass < 1e-6, "causal mask not enforced (future mass not ~0)"

print("All tests passed.")


max attention mass on future positions: 0.0
All tests passed.
